# SQL Injection Detection Model Training

This notebook trains the SQL injection detection models used by the prototype. It focuses only on the machine learning pipeline: dataset loading, preprocessing, TF-IDF feature extraction, Random Forest, XGBoost, Hybrid Ensemble training, evaluation, and artifact export.


## 1. Setup

Install the required packages when running this notebook in a fresh environment. If the packages are already installed, skip the install cell.


In [ ]:
# Optional for fresh environments:
# %pip install pandas scikit-learn xgboost matplotlib seaborn joblib


In [ ]:
from pathlib import Path
from urllib.parse import unquote
import json
import re
import time
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


## 2. Configuration

The dataset file is `Modified_SQL_Dataset.csv` from the Kaggle SQL Injection Dataset by Sajid576. Update `DATASET_PATH` if the file is stored in a different location.


In [ ]:
def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'backend').exists() and (candidate / 'frontend').exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()

DATASET_CANDIDATES = [
    PROJECT_ROOT / 'Modified_SQL_Dataset.csv',
    PROJECT_ROOT / 'data' / 'Modified_SQL_Dataset.csv',
    PROJECT_ROOT / 'backend' / 'data' / 'Modified_SQL_Dataset.csv',
    Path('Modified_SQL_Dataset.csv'),
    Path(r'C:/Users/username/Downloads/Modified_SQL_Dataset.csv/Modified_SQL_Dataset.csv'),
    Path(r'C:/Users/username/Downloads/Modified_SQL_Dataset.csv'),
]

OUTPUT_DIR = PROJECT_ROOT / 'backend' / 'models'
RANDOM_STATE = 42
TEST_SIZE = 0.20

TFIDF_CONFIG = {
    'analyzer': 'char_wb',
    'ngram_range': (2, 4),
    'max_features': 15000,
    'sublinear_tf': True,
    'min_df': 2,
}

print(f'Project root: {PROJECT_ROOT}')
print(f'Output directory: {OUTPUT_DIR}')


## 3. Load Dataset

The dataset must contain the SQL query text and a binary label where `0` means benign and `1` means SQL injection attack. If the query column is named `Query`, it is renamed to `Sentence` for consistency.


In [ ]:
def find_dataset_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        'Modified_SQL_Dataset.csv was not found. Download it from the project Google Drive or Kaggle, then update DATASET_CANDIDATES.'
    )

DATASET_PATH = find_dataset_path(DATASET_CANDIDATES)
df = pd.read_csv(DATASET_PATH)

if 'Query' in df.columns:
    df = df.rename(columns={'Query': 'Sentence'})

required_columns = {'Sentence', 'Label'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

initial_rows = len(df)
print(f'Dataset path: {DATASET_PATH}')
print(f'Initial rows: {initial_rows:,}')
print(df['Label'].value_counts().rename(index={0: 'Benign', 1: 'Attack'}))


## 4. Preprocess and Clean Data

The preprocessing sequence follows the project documentation: URL decoding, string conversion, lowercase normalization, SQL-aware character filtering, whitespace normalization, noisy benign `UNION SELECT` removal, and duplicate removal based on the processed query text.


In [ ]:
PREPROCESS_PATTERN = r"""[^a-zA-Z0-9\s=\<\>\!\'"\(\)\[\]\{\}\-_\+\/\*\&\|\%\.\,\;\:\#\]]+"""

def preprocess_text(text):
    text = unquote(str(text))
    text = text.lower()
    text = re.sub(PREPROCESS_PATTERN, '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

union_noise_mask = df['Sentence'].str.contains('UNION SELECT', case=False, na=False) & (df['Label'] == 0)
removed_noisy_rows = int(union_noise_mask.sum())
df_clean = df.loc[~union_noise_mask].copy()

df_clean['Processed_Sentence'] = df_clean['Sentence'].apply(preprocess_text)
rows_before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Processed_Sentence']).copy()
removed_duplicate_rows = rows_before_dedup - len(df_clean)

print(f'Removed noisy benign UNION SELECT rows: {removed_noisy_rows:,}')
print(f'Removed duplicate processed queries: {removed_duplicate_rows:,}')
print(f'Rows after cleaning: {len(df_clean):,}')
print(df_clean['Label'].value_counts().rename(index={0: 'Benign', 1: 'Attack'}))


## 5. Train-Test Split and TF-IDF Features

The cleaned query text is split using an 80/20 stratified split. TF-IDF is fitted only on the training split to avoid test-set leakage.


In [ ]:
X = df_clean['Processed_Sentence']
y = df_clean['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer = TfidfVectorizer(**TFIDF_CONFIG)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f'Training samples: {X_train_tfidf.shape[0]:,}')
print(f'Test samples: {X_test_tfidf.shape[0]:,}')
print(f'TF-IDF features: {X_train_tfidf.shape[1]:,}')


## 6. Train Models

Three model configurations are trained: Random Forest, XGBoost, and a Hybrid Ensemble using soft voting over Random Forest and XGBoost probabilities.


In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.4f}')

rf_classifier = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_classifier = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

hybrid_model = VotingClassifier(
    estimators=[('rf', rf_classifier), ('xgb', xgb_classifier)],
    voting='soft',
    n_jobs=-1,
)

start = time.time()
rf_classifier.fit(X_train_tfidf, y_train)
print(f'Random Forest trained in {time.time() - start:.2f}s')

start = time.time()
xgb_classifier.fit(X_train_tfidf, y_train)
print(f'XGBoost trained in {time.time() - start:.2f}s')

start = time.time()
hybrid_model.fit(X_train_tfidf, y_train)
print(f'Hybrid Ensemble trained in {time.time() - start:.2f}s')


## 7. Evaluate Models

The models are evaluated using accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrices. These metrics match the structure used in the project documentation.


In [ ]:
def evaluate_model(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_eval, y_pred).ravel()
    return {
        'accuracy': accuracy_score(y_eval, y_pred),
        'precision': precision_score(y_eval, y_pred),
        'recall': recall_score(y_eval, y_pred),
        'f1_score': f1_score(y_eval, y_pred),
        'roc_auc': roc_auc_score(y_eval, y_prob),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

models = {
    'Random Forest': rf_classifier,
    'XGBoost': xgb_classifier,
    'Hybrid Ensemble': hybrid_model,
}

results = {name: evaluate_model(model, X_test_tfidf, y_test) for name, model in models.items()}
results_df = pd.DataFrame(results).T
metric_columns = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
display(results_df[metric_columns].style.format('{:.6f}'))
display(results_df[['tn', 'fp', 'fn', 'tp']].astype(int))


In [ ]:
for name, model in models.items():
    print(f"\n{name} classification report")
    print(classification_report(y_test, model.predict(X_test_tfidf), target_names=['Benign', 'Attack']))


## 8. Visualize Results

The first chart compares model performance. The confusion matrices show benign and attack classification outcomes for each model.


In [ ]:
plot_df = results_df[metric_columns].rename(columns={'f1_score': 'f1-score', 'roc_auc': 'roc-auc'})
ax = plot_df.plot(kind='bar', figsize=(10, 5), ylim=(0.98, 1.0), rot=0)
ax.set_title('Model Performance Comparison')
ax.set_ylabel('Score')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cmaps = ['Blues', 'Oranges', 'Greens']

for ax, (name, model), cmap in zip(axes, models.items(), cmaps):
    cm = confusion_matrix(y_test, model.predict(X_test_tfidf))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap=cmap,
        cbar=False,
        xticklabels=['Benign', 'Attack'],
        yticklabels=['Benign', 'Attack'],
        ax=ax,
    )
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('Actual Label')

plt.tight_layout()
plt.show()


## 9. Export Model Artifacts

The exported artifacts are used by the FastAPI backend: the hybrid model, TF-IDF vectorizer, Random Forest model, XGBoost model, and metadata files.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(hybrid_model, OUTPUT_DIR / 'hybrid_model.pkl')
joblib.dump(vectorizer, OUTPUT_DIR / 'tfidf_vectorizer.pkl')
joblib.dump(rf_classifier, OUTPUT_DIR / 'rf_model.pkl')
joblib.dump(xgb_classifier, OUTPUT_DIR / 'xgb_model.pkl')

metadata = {
    'source_notebook': 'notebooks/training_workspace/SQLI_Model_Training_Clean.ipynb',
    'dataset_name': 'Modified_SQL_Dataset / 30k dataset',
    'dataset_path': str(DATASET_PATH),
    'exported_at': datetime.now(timezone.utc).isoformat(),
    'initial_rows': int(initial_rows),
    'removed_noisy_benign_union_select_rows': int(removed_noisy_rows),
    'removed_duplicate_processed_queries': int(removed_duplicate_rows),
    'rows_after_duplicate_removal': int(len(df_clean)),
    'test_size': TEST_SIZE,
    'random_state': RANDOM_STATE,
    'stratified': True,
    'tfidf': {
        'analyzer': TFIDF_CONFIG['analyzer'],
        'ngram_range': list(TFIDF_CONFIG['ngram_range']),
        'max_features': TFIDF_CONFIG['max_features'],
        'sublinear_tf': TFIDF_CONFIG['sublinear_tf'],
        'min_df': TFIDF_CONFIG['min_df'],
    },
    'random_forest': {
        'n_estimators': 200,
        'class_weight': 'balanced',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
    },
    'xgboost': {
        'n_estimators': 200,
        'max_depth': 6,
        'learning_rate': 0.1,
        'scale_pos_weight': float(scale_pos_weight),
        'eval_metric': 'logloss',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
    },
    'metrics': {
        name.lower().replace(' ', '_'): {
            key: (float(value) if key in metric_columns else int(value))
            for key, value in row.items()
        }
        for name, row in results_df.to_dict(orient='index').items()
    },
}

(OUTPUT_DIR / 'training_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print(f'Artifacts exported to: {OUTPUT_DIR.resolve()}')
print('Saved: hybrid_model.pkl, tfidf_vectorizer.pkl, rf_model.pkl, xgb_model.pkl, training_metadata.json, metrics.json')


## 10. Quick Prediction Check

This final cell confirms that the exported pipeline can transform a SQL query and return a prediction.


In [ ]:
sample_queries = [
    "SELECT id, username FROM users WHERE username = 'maria';",
    "' OR 1=1 /*",
    "UNION SELECT NULL, table_name FROM information_schema.tables",
    "1; DROP TABLE users; --",
]

for query in sample_queries:
    processed = preprocess_text(query)
    vector = vectorizer.transform([processed])
    prediction = hybrid_model.predict(vector)[0]
    probability = hybrid_model.predict_proba(vector)[0][1]
    label = 'Attack' if prediction == 1 else 'Benign'
    print(f'{label:<8} attack_probability={probability:.4f} | {query}')
